# 02b · PyTorch Basics & the Training Loop `[local]`

**Goal:** rebuild 02a's network in PyTorch, understand every line of the standard training loop, and set up `cvlib.train`, which every later module reuses.

| Section | What you'll do |
|---|---|
| 🧠 Intuition | PyTorch = NumPy + automatic gradients + GPU |
| 📐 Math | Computational graphs, why gradients accumulate, SGD → momentum → Adam |
| 🔨 From scratch | Tensors, autograd by hand, the 02a network with raw tensors |
| 🚀 Real tool | `nn.Module`, `DataLoader`, optimizers, `cvlib.train.fit`, CPU vs. GPU |
| 📊 Evaluate | Test accuracy, confusion matrix, save/load check, early stopping |
| 🧪 Experiment | Batch size, dropout, depth |

In [ ]:
# ── Standard first cell ──
import sys, pathlib
for _p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents, *pathlib.Path("/kaggle/input").glob("*")]:
    if (_p / "cvlib").is_dir():
        sys.path.insert(0, str(_p)); break
from cvlib.env import setup
env = setup()

import time
import inspect
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, random_split
from torchvision import datasets, transforms
from cvlib import train as cvtrain
from cvlib.viz import plot_history

device = env.device
print("using device:", device)

---
# 🧠 1 · Intuition

PyTorch gives you three things on top of NumPy:

1. **Tensors**: NumPy-like arrays that can live on a **GPU** (`.to("mps")` or `.to("cuda")`).
2. **Autograd**: every operation on a tensor with `requires_grad=True` is **recorded**. Calling `.backward()` replays the recording in reverse and computes every gradient: the backprop you wrote by hand in 02a, automated.
3. **Building blocks**: layers (`nn.Linear`, `nn.Conv2d`, ...), losses, optimizers, data loaders.

Since autograd works for *any* combination of operations, you can invent new architectures without deriving a single gradient. That's why the field moves so fast.

---
# 📐 2 · The math

### Computational graph
The forward pass $x \to z = xW + b \to a = \text{ReLU}(z) \to \dots \to L$ is recorded as a graph. `L.backward()` walks it backwards, applying the chain rule at each node:

$$\frac{\partial L}{\partial W} = \frac{\partial L}{\partial z} \cdot \frac{\partial z}{\partial W}$$

### Gradients accumulate
`.backward()` **adds** to `.grad` instead of overwriting it (useful for summing gradients over several small batches). That's why every training step starts with `optimizer.zero_grad()`. Forgetting it is a classic bug.

### Optimizers: smarter steps downhill
- **SGD:** $w \leftarrow w - \eta\, g$
- **SGD + momentum:** keep a running velocity, like a ball rolling downhill. It speeds up along consistent directions and dampens zig-zags.
$$v \leftarrow \beta v + g, \qquad w \leftarrow w - \eta\, v \qquad (\beta \approx 0.9)$$
- **Adam:** momentum **plus** a per-weight step size, scaled by how large that weight's gradients usually are. Weights with consistently small gradients get relatively bigger steps. Works well with little tuning (`lr=1e-3`), which makes it the usual first choice.

---
# 🔨 3 · From scratch with tensors

### 3.1 · Tensors in 60 seconds

In [ ]:
a = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
b = torch.ones(2, 2)
print("a @ b =\n", a @ b)
print("shape:", a.shape, "| dtype:", a.dtype, "| device:", a.device)

n = a.numpy()                                   # tensor → NumPy (shares memory on the CPU!)
t = torch.from_numpy(np.arange(6.0)).reshape(2, 3)
print("from numpy:", t)

g = a.to(device)                                # move to the GPU
print("on", g.device, "| back to cpu:", g.cpu().device)
# Rule: tensors in one operation must be on the same device. "Expected all tensors to be on the same device" = you forgot a .to(device).

### 3.2 · Autograd by hand: $f(x) = x^2 + 3x$, so $f'(x) = 2x + 3$

In [ ]:
x = torch.tensor(2.0, requires_grad=True)       # "record what happens to x"
f = x ** 2 + 3 * x
f.backward()                                    # compute df/dx
print(f"f(2) = {f.item()} | autograd df/dx = {x.grad.item()} | by hand 2·2+3 = {2 * 2 + 3}")

f = x ** 2 + 3 * x
f.backward()
print("called backward again without zeroing → grad =", x.grad.item(), "(7 + 7: accumulated!)")
x.grad.zero_();                                 # reset (the ; hides the printed return value)

### 3.3 · The 02a network, with raw tensors and manual updates

No `nn.Module`, no optimizer. Just tensors and autograd, to see exactly what the high-level tools do.

In [ ]:
train_full = datasets.MNIST(env.data_dir, train=True, download=True)
test_raw = datasets.MNIST(env.data_dir, train=False, download=True)

X_all = train_full.data.float().div(255).reshape(-1, 784)
y_all = train_full.targets
g_split = torch.Generator().manual_seed(42)
perm = torch.randperm(60_000, generator=g_split)
X_tr, y_tr = X_all[perm[:50_000]], y_all[perm[:50_000]]
X_va, y_va = X_all[perm[50_000:]], y_all[perm[50_000:]]
X_te, y_te = test_raw.data.float().div(255).reshape(-1, 784), test_raw.targets

torch.manual_seed(0)
W1 = (torch.randn(784, 128) * (2 / 784) ** 0.5).requires_grad_()     # He init, as in 02a
b1 = torch.zeros(128, requires_grad=True)
W2 = (torch.randn(128, 10) * (2 / 128) ** 0.5).requires_grad_()
b2 = torch.zeros(10, requires_grad=True)
params = [W1, b1, W2, b2]

lr, bs = 0.1, 64
for epoch in range(3):
    order = torch.randperm(len(X_tr))
    for s in range(0, len(X_tr), bs):
        idx = order[s:s + bs]
        logits = torch.relu(X_tr[idx] @ W1 + b1) @ W2 + b2     # forward
        loss = F.cross_entropy(logits, y_tr[idx])               # softmax + cross-entropy
        loss.backward()                                          # autograd = your 02a backward()
        with torch.no_grad():                                    # the update itself must not be recorded
            for p in params:
                p -= lr * p.grad
                p.grad.zero_()                                   # don't let gradients accumulate
    with torch.no_grad():
        val_acc = ((torch.relu(X_va @ W1 + b1) @ W2 + b2).argmax(1) == y_va).float().mean()
    print(f"epoch {epoch + 1}: val acc {val_acc:.4f}")

Same ~95–96% after 3 epochs as 02a, and you didn't write a backward pass.

---
# 🚀 4 · The PyTorch way

### 4.1 · `nn.Module`: a model as a reusable object

In [ ]:
class MLP(nn.Module):
    def __init__(self, hidden=128, n_layers=1, dropout=0.0):
        super().__init__()
        layers, d = [nn.Flatten()], 784                          # (B, 1, 28, 28) → (B, 784)
        for _ in range(n_layers):
            layers += [nn.Linear(d, hidden), nn.ReLU(), nn.Dropout(dropout)]
            d = hidden
        layers.append(nn.Linear(d, 10))                          # logits: NO softmax here,
        self.net = nn.Sequential(*layers)                        # CrossEntropyLoss applies it internally

    def forward(self, x):
        return self.net(x)

model = MLP()
print(model)
print(f"parameters: {sum(p.numel() for p in model.parameters()):,}  (same 101,770 as 02a)")

⚠️ **Don't put a softmax at the end** when using `nn.CrossEntropyLoss`. It expects raw scores (logits) and applies log-softmax itself, in a numerically safer way.

### 4.2 · `Dataset` and `DataLoader`

A **Dataset** answers two questions: how many items, and what is item *i*. A **DataLoader** wraps it to deliver shuffled **batches**.

`transforms` convert each image on the fly. `ToTensor()` turns a 28×28 PIL image into a `(1, 28, 28)` float tensor in 0..1 (channels first!), and `Normalize` shifts it to mean 0, std 1.

In [ ]:
tfm = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])  # MNIST mean/std
mnist_train = datasets.MNIST(env.data_dir, train=True, download=True, transform=tfm)
mnist_test = datasets.MNIST(env.data_dir, train=False, download=True, transform=tfm)
train_ds, val_ds = random_split(mnist_train, [50_000, 10_000], generator=torch.Generator().manual_seed(42))

loader = DataLoader(train_ds, batch_size=64, shuffle=True)
xb, yb = next(iter(loader))
print("one batch:", xb.shape, xb.dtype, "| labels:", yb.shape, yb[:10].tolist())
print(f"pixel mean {xb.mean():.2f}, std {xb.std():.2f} (normalized)")

**Batch shape `(B, C, H, W)`** = (batch, channels, height, width) is PyTorch's convention for images. OpenCV uses `(H, W, C)`. Converting between the two is something you'll do constantly.

**Speed tip:** decoding images one at a time in Python is slow. MNIST fits in memory, so we pre-convert it once and use a `TensorDataset`. Same data, much faster.

In [ ]:
def to_tensors(ds):
    xs, ys = zip(*[ds[i] for i in range(len(ds))])
    return torch.stack(xs), torch.tensor(ys)

t0 = time.perf_counter()
Xtr, Ytr = to_tensors(train_ds); Xva, Yva = to_tensors(val_ds); Xte, Yte = to_tensors(mnist_test)
print(f"pre-converted in {time.perf_counter() - t0:.1f}s: train {tuple(Xtr.shape)}, val {tuple(Xva.shape)}, test {tuple(Xte.shape)}")

def loaders(batch_size=128):
    return (DataLoader(TensorDataset(Xtr, Ytr), batch_size=batch_size, shuffle=True),
            DataLoader(TensorDataset(Xva, Yva), batch_size=1024),
            DataLoader(TensorDataset(Xte, Yte), batch_size=1024))

train_loader, val_loader, test_loader = loaders()

### 4.3 · The training loop, now in `cvlib.train`

These ~20 lines are the heart of almost every PyTorch project. Read them once carefully:

In [ ]:
print(inspect.getsource(cvtrain.train_one_epoch))

`model.train()` / `model.eval()` switch layers like **Dropout** and **BatchNorm** between training and inference behavior. `@torch.no_grad()` on evaluation skips graph recording. `fit()` wraps both with history tracking and optional **early stopping**.

### 4.4 · CPU or GPU? Measure, don't assume

In [ ]:
def time_epoch(dev, batch_size):
    m = MLP().to(dev)
    opt = torch.optim.SGD(m.parameters(), lr=0.1)
    tl, _, _ = loaders(batch_size)
    cvtrain.train_one_epoch(m, tl, nn.CrossEntropyLoss(), opt, dev)          # warm-up epoch
    t0 = time.perf_counter()
    cvtrain.train_one_epoch(m, tl, nn.CrossEntropyLoss(), opt, dev)
    return time.perf_counter() - t0

for bs in (64, 1024):
    line = f"batch {bs:>4}: cpu {time_epoch('cpu', bs):.2f}s/epoch"
    if device != "cpu":
        line += f" | {device} {time_epoch(device, bs):.2f}s/epoch"
    print(line)

For a model this tiny, the GPU often **isn't** faster: each batch is so little work that the overhead of launching GPU operations dominates. The GPU wins when there's lots of math per step: big batches, convolutions, large models. From module 03 on, it wins clearly.

### 4.5 · Train it: SGD vs. SGD + momentum vs. Adam

In [ ]:
results = {}
for name, make_opt in {
    "SGD":          lambda p: torch.optim.SGD(p, lr=0.05),
    "SGD+momentum": lambda p: torch.optim.SGD(p, lr=0.05, momentum=0.9),
    "Adam":         lambda p: torch.optim.Adam(p, lr=1e-3),
}.items():
    torch.manual_seed(0)
    m = MLP()
    print(f"── {name}")
    results[name] = (m, cvtrain.fit(m, train_loader, val_loader, nn.CrossEntropyLoss(), make_opt(m.parameters()), epochs=5, device="cpu"))

plt.figure(figsize=(7, 3.5))
for name, (_, h) in results.items():
    plt.plot(range(1, 6), h["val_acc"], marker="o", label=name)
plt.xlabel("epoch"); plt.ylabel("val accuracy"); plt.legend(); plt.grid(alpha=0.3); plt.title("Optimizers compared"); plt.show()

Momentum and Adam reach high accuracy much faster than plain SGD at the same learning rate. In practice: **Adam (lr 1e-3) to get going quickly**; well-tuned SGD + momentum is still common for training CNNs from scratch. You'll compare them again in module 05.

---
# 📊 5 · Evaluate

### 5.1 · Test accuracy and confusion matrix

Pick the model using **validation** accuracy, then evaluate it on test **once**.

In [ ]:
best_name = max(results, key=lambda k: results[k][1]["val_acc"][-1])
best_model = results[best_name][0]
logits, labels = cvtrain.predict(best_model, test_loader, "cpu")
preds = logits.argmax(1)
print(f"chosen by validation: {best_name} | TEST accuracy: {(preds == labels).float().mean():.4f}")

cm = torch.zeros(10, 10, dtype=torch.long)
cm.index_put_((labels, preds), torch.ones_like(labels), accumulate=True)
per_class = (cm.diagonal() / cm.sum(1)).numpy()
print("per-class accuracy:", " ".join(f"{d}:{a:.3f}" for d, a in enumerate(per_class)))
off = cm.clone().fill_diagonal_(0)
i, j = divmod(int(off.argmax()), 10)
print(f"most common confusion: {i} → {j} ({off[i, j]} times)")

### 5.2 · Save, load, and check you get the same answers

Save the **`state_dict`** (the weights), not the whole model object, which is fragile across code changes. Always verify a reload reproduces the predictions.

In [ ]:
path = env.out_dir / "mlp_mnist.pt"
torch.save(best_model.state_dict(), path)

reloaded = MLP()
reloaded.load_state_dict(torch.load(path, weights_only=True))
logits2, _ = cvtrain.predict(reloaded, test_loader, "cpu")
print(f"saved {path.stat().st_size / 1e3:.0f} KB | identical predictions after reload: {torch.equal(logits.argmax(1), logits2.argmax(1))}")

### 5.3 · Early stopping in action

Recreate 02a's overfitting setup (1,000 training images, a big network, many epochs). With `patience=5`, `fit` stops when validation loss stops improving and **restores the best weights**.

In [ ]:
torch.manual_seed(0)
small_loader = DataLoader(TensorDataset(Xtr[:1000], Ytr[:1000]), batch_size=32, shuffle=True)
m_es = MLP(hidden=512)
h_es = cvtrain.fit(m_es, small_loader, val_loader, nn.CrossEntropyLoss(), torch.optim.Adam(m_es.parameters(), 1e-3),
                   epochs=100, device="cpu", patience=5, verbose=False)
best_epoch = int(np.argmin(h_es["val_loss"])) + 1
print(f"stopped after {len(h_es['val_loss'])} of 100 epochs | best epoch {best_epoch} | val acc there: {h_es['val_acc'][best_epoch - 1]:.4f}")
plot_history(h_es)

---
# 🧪 6 · Experiments: predict first, then run

**Experiment 1: batch size.** With batch sizes 16, 128 and 1024 at the same learning rate and 3 epochs, predict which is fastest per epoch and which reaches the best accuracy.

In [ ]:
# Prediction: ...
for bs in [16, 128, 1024]:
    torch.manual_seed(0)
    m = MLP(); tl, vl, _ = loaders(bs)
    t0 = time.perf_counter()
    h = cvtrain.fit(m, tl, vl, nn.CrossEntropyLoss(), torch.optim.Adam(m.parameters(), 1e-3), epochs=3, device="cpu", verbose=False)
    print(f"batch {bs:>4}: {len(tl):>4} steps/epoch | {(time.perf_counter() - t0) / 3:.2f}s/epoch | val acc {h['val_acc'][-1]:.4f}")
# Big batches: fewer, more efficient steps (fast per epoch) but fewer updates in total, so they learn
# less per epoch at the same lr. Small batches: many noisy updates, slow per epoch. Common fix for
# big batches: raise the learning rate.

**Experiment 2: dropout against overfitting.** Dropout randomly zeroes a fraction of neurons during training, so the network can't rely on any single one. On the 1,000-image subset, predict the effect of `dropout=0.5`.

In [ ]:
# Prediction: ...
for p in [0.0, 0.5]:
    torch.manual_seed(0)
    m = MLP(hidden=512, dropout=p)
    h = cvtrain.fit(m, small_loader, val_loader, nn.CrossEntropyLoss(), torch.optim.Adam(m.parameters(), 1e-3), epochs=30, device="cpu", verbose=False)
    print(f"dropout {p}: train acc {h['train_acc'][-1]:.3f} | best val acc {max(h['val_acc']):.4f} | final val loss {h['val_loss'][-1]:.3f}")
# Only a small gain here: with just 1,000 images, no single trick fixes overfitting.
# Dropout earns its keep combined with data augmentation and weight decay (module 05).

**Experiment 3: deeper vs. wider.** Compare 1 hidden layer of 128, 3 layers of 128, and 1 layer of 512. Predict which wins on MNIST.

In [ ]:
# Prediction: ...
for desc, kw in [("1 × 128", dict(hidden=128, n_layers=1)), ("3 × 128", dict(hidden=128, n_layers=3)), ("1 × 512", dict(hidden=512, n_layers=1))]:
    torch.manual_seed(0)
    m = MLP(**kw)
    h = cvtrain.fit(m, train_loader, val_loader, nn.CrossEntropyLoss(), torch.optim.Adam(m.parameters(), 1e-3), epochs=5, device="cpu", verbose=False)
    print(f"{desc}: {sum(p.numel() for p in m.parameters()):>7,} params | val acc {h['val_acc'][-1]:.4f}")
# All land around 97% after 5 epochs. Extra size helps a little, extra depth barely at all: fully-connected
# networks hit a ceiling on images because they ignore that
# nearby pixels are related. Module 03's CNNs break through it with fewer parameters.

---
# 📝 7 · Recap

**Cheat sheet**
- Tensor = NumPy array + device + autograd. Keep everything in one operation on the same device.
- `requires_grad=True` records operations; `loss.backward()` fills `.grad`. Gradients **accumulate**, so call `zero_grad()` every step.
- The loop: `model.train()` → forward → loss → `zero_grad()` → `backward()` → `step()`. Evaluate under `model.eval()` + `torch.no_grad()`.
- `nn.CrossEntropyLoss` takes **logits** (no softmax in the model).
- Images in PyTorch are `(B, C, H, W)`; OpenCV is `(H, W, C)`.
- Adam (1e-3) to start; SGD + momentum is also strong. Measure whether the GPU actually helps.
- Save `state_dict`, reload, verify. Early stopping keeps the best-validation weights.
- Reusable: `from cvlib.train import fit, evaluate, predict`.

**Interview-style questions**
1. What does `optimizer.zero_grad()` do, and what happens if you forget it?
2. Why wrap evaluation in `torch.no_grad()`? What do `model.train()` and `model.eval()` change?
3. Why shouldn't a model end with softmax when trained with `CrossEntropyLoss`?
4. Explain momentum with an analogy. What does Adam add on top?
5. When would training on a GPU be *slower* than on the CPU?
6. How does batch size affect speed, memory and the number of updates per epoch?
7. How does early stopping decide when to stop, and which weights does it keep?

---
# 🎉 Phase 1 complete

You can now handle images with OpenCV (01) and build, train and evaluate neural networks (02). Phase 2 combines them: **convolutional neural networks**, where the convolution from 01b becomes a learned layer.

**Next:** module 03, *CNNs in depth*.